In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from scipy.optimize import fsolve, minimize
from scipy.integrate import quad
from numpy import linspace, meshgrid, arange, empty, concatenate, newaxis, shape
from collections import deque

%run estimation.ipynb
%run MDP_core_allocation.ipynb

In [ ]:
def simulate_until_n_departures(jobs, jobs_in_system_ids, current_time, n_departures, amdahl_pars, model_pars, pi_optimal, M):
    """
    Simulate queue until exactly n_departures occur, continuing from existing jobs in system
    
    Parameters:
    - jobs: list of ALL Job objects (already created with arrival/service times, categories)
    - jobs_in_system_ids: list of job IDs currently in system (from previous window)
    - current_time: current simulation time
    - n_departures: number of departures to wait for in this window
    - amdahl_pars: [p1, p2] for simulation (TRUE parameters)
    - model_pars: [lam, mu, cores, alpha]
    - pi_optimal: optimal policy matrix
    - M: max queue size for policy lookup
    
    Returns: events, n_jobs_after_events, core_alloc_after_events, jobs_in_system_ids, current_time
    """
    lam, mu, total_cores, alpha = model_pars
    p1, p2 = amdahl_pars
    
    events = []
    n_jobs_in_system_just_after_events = []
    core_allocation_just_after_events = []
    
    job_ids_in_system = jobs_in_system_ids.copy()
    departure_count = 0
    
    next_arrival_id = None
    for jid in range(len(jobs)):
        if jobs[jid].arrival_time >= current_time and jid not in job_ids_in_system:
            next_arrival_id = jid
            break
    
    n_jobs_in_system = np.array([
        sum([1 for jid in job_ids_in_system if jobs[jid].category == 1]),
        sum([1 for jid in job_ids_in_system if jobs[jid].category == 2])
    ])
    
    while departure_count < n_departures:
        if next_arrival_id is not None and next_arrival_id < len(jobs):
            time_until_next_arrival = jobs[next_arrival_id].arrival_time - current_time
        else:
            time_until_next_arrival = math.inf
        
        if len(job_ids_in_system) == 0:
            time_until_next_departure = math.inf
        else:
            n1, n2 = min(n_jobs_in_system[0], M), min(n_jobs_in_system[1], M)
            fraction_to_type1 = pi_optimal[n1, n2]
            current_core_allocation = np.array([fraction_to_type1 * total_cores, (1 - fraction_to_type1) * total_cores])
            
            cores_per_job = np.array([
                current_core_allocation[0]/n_jobs_in_system[0] if n_jobs_in_system[0] > 0 else 0,
                current_core_allocation[1]/n_jobs_in_system[1] if n_jobs_in_system[1] > 0 else 0
            ])
            speed_up_values = np.array([speed_up(p1, cores_per_job[0]), speed_up(p2, cores_per_job[1])])
            
            expected_times = np.array([
                jobs[jid].residual_service_time / speed_up_values[int(jobs[jid].category) - 1]
                for jid in job_ids_in_system
            ])
            time_until_next_departure = np.min(expected_times)
        
        if time_until_next_arrival == math.inf and time_until_next_departure == math.inf:
            break
        
        if time_until_next_arrival < time_until_next_departure:
            time_step = time_until_next_arrival
            event_type = 'arrival'
            event_job_id = next_arrival_id
        else:
            time_step = time_until_next_departure
            event_type = 'departure'
            event_job_id = job_ids_in_system[np.argmin(expected_times)]
        
        if len(job_ids_in_system) > 0:
            for jid in job_ids_in_system:
                spd = speed_up_values[int(jobs[jid].category) - 1]
                jobs[jid].residual_service_time -= spd * time_step
                jobs[jid].history.append([spd, time_step])
        
        current_time += time_step
        
        if event_type == 'arrival':
            events.append([current_time, 'arrival', event_job_id])
            job_ids_in_system.append(event_job_id)
            n_jobs_in_system[int(jobs[event_job_id].category) - 1] += 1
            
            next_arrival_id = None
            for jid in range(event_job_id + 1, len(jobs)):
                if jobs[jid].arrival_time >= current_time and jid not in job_ids_in_system:
                    next_arrival_id = jid
                    break
        else:
            events.append([current_time, 'departure', event_job_id])
            job_ids_in_system.remove(event_job_id)
            n_jobs_in_system[int(jobs[event_job_id].category) - 1] -= 1
            jobs[event_job_id].departure_time = current_time
            departure_count += 1
        
        n_jobs_in_system_just_after_events.append(n_jobs_in_system.copy())
        n1, n2 = min(n_jobs_in_system[0], M), min(n_jobs_in_system[1], M)
        fraction = pi_optimal[n1, n2]
        core_allocation_just_after_events.append(np.array([fraction * total_cores, (1-fraction) * total_cores]))
    
    return events, n_jobs_in_system_just_after_events, core_allocation_just_after_events, job_ids_in_system, current_time

In [ ]:
def iterative_algorithm_changing_p(base_pars, initial_estimates, N, total_iterations, p_schedule):
    """
    Algorithm 1 variant where the TRUE speed-up parameters change at specified iterations.
    
    Parameters:
    - base_pars: [lam, mu, cores, alpha, M] (everything except p1, p2)
    - initial_estimates: [p1_est_0, p2_est_0]
    - N: base window size
    - total_iterations: total number of iterations
    - p_schedule: list of (iteration, p1_true, p2_true) tuples, sorted by iteration.
                  The first entry should have iteration=1.
    
    Returns: estimates_history, policies_history, p_true_history
    """
    lam, mu, cores, alpha, M = base_pars
    model_pars = [lam, mu, cores, alpha]
    
    p1_true, p2_true = p_schedule[0][1], p_schedule[0][2]
    schedule_idx = 0
    
    p1_est, p2_est = initial_estimates
    estimates_history = [initial_estimates]
    policies_history = []
    p_true_history = [(p1_true, p2_true)]
    
    exponent = 0.75
    total_departures_estimate = sum([int(N * k) for k in range(1, total_iterations + 1)])
    total_jobs = int(total_departures_estimate * 2)
    
    arrival_times = generate_arrival_times(lam, total_jobs)
    service_times = generate_service_times(mu, total_jobs)
    job_categories = generate_job_categories(alpha, total_jobs)
    
    jobs = [Job() for i in range(total_jobs)]
    for i in range(total_jobs):
        jobs[i].identity = i
        jobs[i].arrival_time = arrival_times[i]
        jobs[i].service_time = service_times[i]
        jobs[i].residual_service_time = service_times[i]
        jobs[i].category = int(job_categories[i])
    
    jobs_in_system_ids = []
    current_time = 0.0
    
    print("="*70)
    print("ALGORITHM 1: CHANGING SPEED-UP PARAMETERS")
    print("="*70)
    print(f"Initial estimates: p1 = {p1_est:.4f}, p2 = {p2_est:.4f}")
    print(f"Parameter schedule:")
    for it, p1, p2 in p_schedule:
        print(f"  From iteration {it}: p1 = {p1:.4f}, p2 = {p2:.4f}")
    print("="*70)
    
    for iteration in range(1, total_iterations + 1):
        if schedule_idx + 1 < len(p_schedule) and iteration >= p_schedule[schedule_idx + 1][0]:
            schedule_idx += 1
            p1_true = p_schedule[schedule_idx][1]
            p2_true = p_schedule[schedule_idx][2]
            print(f"\n*** PARAMETER CHANGE at iteration {iteration}: "
                  f"p1 = {p1_true:.4f}, p2 = {p2_true:.4f} ***")
        
        amdahl_pars_true = [p1_true, p2_true]
        
        n_departures = int(N * np.power(iteration, exponent))
        
        print(f"\n--- Iteration {iteration}/{total_iterations} ---")
        print(f"True: p1 = {p1_true:.4f}, p2 = {p2_true:.4f} | "
              f"Estimates: p1 = {p1_est:.4f}, p2 = {p2_est:.4f}")
        
        est_pars = [lam, mu, cores, p1_est, p2_est, alpha, M]
        pi_optimal, V_optimal, Q_optimal = bellman_optimal_policy(est_pars)
        policies_history.append(pi_optimal)
        
        events, n_jobs, core_alloc, jobs_in_system_ids, current_time = simulate_until_n_departures(
            jobs, jobs_in_system_ids, current_time, n_departures,
            amdahl_pars_true, model_pars, pi_optimal, M
        )
        
        if len(events) == 0:
            print("WARNING: No events generated! Keeping previous estimates.")
            estimates_history.append((p1_est, p2_est))
            p_true_history.append((p1_true, p2_true))
            continue
        
        estd_result = estimation(
            model_pars, jobs, jobs_in_system_ids,
            events, n_jobs, core_alloc
        )
        
        p1_est, p2_est = estd_result.x
        p1_est = np.clip(p1_est, 0.001, 0.999)
        p2_est = np.clip(p2_est, 0.001, 0.999)
        estimates_history.append((p1_est, p2_est))
        p_true_history.append((p1_true, p2_true))
        
        error_p1 = abs(p1_est - p1_true)
        error_p2 = abs(p2_est - p2_true)
        print(f"Updated estimates: p1 = {p1_est:.4f}, p2 = {p2_est:.4f} | "
              f"Errors: \u0394p1 = {error_p1:.4f}, \u0394p2 = {error_p2:.4f}")
    
    print("\n" + "="*70)
    print("ALGORITHM COMPLETED")
    print("="*70)
    
    return estimates_history, policies_history, p_true_history

In [ ]:
lam = 2
mu = 1.5
cores = 10
alpha = 0.7
M = 10

base_pars = [lam, mu, cores, alpha, M]
initial_estimates = [0.5, 0.5]
N = 50
total_iterations = 60

p_schedule = [
    (1,  0.40, 0.85),
    (21, 0.70, 0.50),
    (41, 0.25, 0.70),
]

estimates_hist_cp, policies_hist_cp, p_true_hist_cp = iterative_algorithm_changing_p(
    base_pars, initial_estimates, N, total_iterations, p_schedule
)

In [ ]:
p1_est = [e[0] for e in estimates_hist_cp]
p2_est = [e[1] for e in estimates_hist_cp]
p1_true_vals = [t[0] for t in p_true_hist_cp]
p2_true_vals = [t[1] for t in p_true_hist_cp]
iterations = range(len(p1_est))

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(iterations, p1_est, 'o-', linewidth=2, markersize=4, label=r'$\widehat{p}_{1,k}$')
ax1.plot(iterations, p1_true_vals, 'r--', linewidth=2, label=r'True $p_1$')
for it, p1v, _ in p_schedule[1:]:
    ax1.axvline(x=it, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
ax1.set_xlabel(r'$k$', fontsize=20)
ax1.set_ylabel(r'$\widehat{p}_{1,k}$', fontsize=20)
ax1.legend(fontsize=14)
ax1.grid(True, alpha=0.3)

ax2.plot(iterations, p2_est, 'o-', linewidth=2, markersize=4, color='green', label=r'$\widehat{p}_{2,k}$')
ax2.plot(iterations, p2_true_vals, 'r--', linewidth=2, label=r'True $p_2$')
for it, _, p2v in p_schedule[1:]:
    ax2.axvline(x=it, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
ax2.set_xlabel(r'$k$', fontsize=20)
ax2.set_ylabel(r'$\widehat{p}_{2,k}$', fontsize=20)
ax2.legend(fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('convergence_changing_p.png', dpi=300)
plt.show()